# 09 Persistence test (T24)
Does using the previous hour make the detector miss long-lasting anomalies? Run after notebook 08 (needs flags_ext.csv.gz). Compares events found by event length at 1% of hours flagged, with 95% intervals grouped by building, and the share of hours flagged at each position inside events longer than 12 hours.

In [ ]:
import sys
from config import *
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
RATE=1.0; NB=2000
METH={"last_hour":"Same as last hour","typical":"Historical typical value","C":"Model C","iforest_raw":"Isolation Forest (raw)"}
LG=["1 h","2-3 h","4-12 h",">12 h"]
f=pd.read_csv(OUT_DIR/"flags_ext.csv.gz",parse_dates=["timestamp"])
f=f[f.split=="test"].sort_values(["building_id","timestamp"]).reset_index(drop=True)
f["t"]=(f.timestamp-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
ev=pd.read_csv(OUT_DIR/"lead_events.csv",parse_dates=["start","end"]);ev["split"]=ev["split"].replace({"validation":"val"})
ev=ev[(ev.split=="test")&(ev.zero_share<=0.5)&ev.building_id.isin(f.building_id.unique())].copy()
ev["s"]=(ev.start-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1);ev["e"]=(ev.end-pd.Timestamp("2016-01-01"))//pd.Timedelta(hours=1)
vt={b:g.t.to_numpy() for b,g in f[f.valid].groupby("building_id")}
ev["nv"]=[0 if vt.get(b) is None else int(np.searchsorted(vt[b],e,"right")-np.searchsorted(vt[b],s,"left")) for b,s,e in zip(ev.building_id,ev.s,ev.e)]
ev=ev[ev.nv>0].reset_index(drop=True)
lgrp=ev["length_group"].astype(str).to_numpy()
def mask(score,mode):
    s=pd.Series(score,index=f.index).where(f.valid)
    if mode=="pooled": return (s>np.nanquantile(s.to_numpy(),1-RATE/100)).fillna(False).to_numpy()
    p=s.groupby(f.building_id).rank(pct=True,method="first"); return (p>1-RATE/100).fillna(False).to_numpy()
bl=ev.building_id.to_numpy();ub=np.unique(bl);ib={b:np.where(bl==b)[0] for b in ub}
rng=np.random.default_rng(0);boots=[np.concatenate([ib[b] for b in rng.choice(ub,len(ub))]) for _ in range(NB)]
rows=[];diffs=[];pos=[]
for mode in ["per_building","pooled"]:
    found={};flagset={}
    for m in METH:
        mk=mask(f["score_"+m].to_numpy(),mode)
        T={b:g.t.to_numpy() for b,g in f[mk].groupby("building_id")}
        fo=np.array([(lambda t:False if t is None else np.searchsorted(t,e,"right")>np.searchsorted(t,s,"left"))(T.get(b)) for b,s,e in zip(ev.building_id,ev.s,ev.e)])
        found[m]=fo;flagset[m]=T
        for g in LG:
            gi=np.where(lgrp==g)[0]
            bs=[]
            for bi in boots:
                x=bi[lgrp[bi]==g]
                if len(x): bs.append(fo[x].mean()*100)
            rows.append(dict(mode=mode,length=g,method=METH[m],events=len(gi),found=int(fo[gi].sum()),recall_pct=round(fo[gi].mean()*100,1),ci_low=round(np.percentile(bs,2.5),1),ci_high=round(np.percentile(bs,97.5),1)))
        # share of event hours flagged by position in event (event hour 1, 2-3, 4-12, >12), events longer than 12 h
    for a,b in [("C","last_hour"),("C","typical")]:
        for g in LG:
            gi=np.where(lgrp==g)[0]
            d0=(found[a][gi].mean()-found[b][gi].mean())*100
            bs=[]
            for bi in boots:
                x=bi[lgrp[bi]==g]
                if len(x): bs.append((found[a][x].mean()-found[b][x].mean())*100)
            diffs.append(dict(mode=mode,length=g,comparison=f"{METH[a]} minus {METH[b]}",diff_pp=round(d0,1),ci_low=round(np.percentile(bs,2.5),1),ci_high=round(np.percentile(bs,97.5),1)))
    if mode=="per_building":
        long_i=np.where(lgrp==">12 h")[0]
        for m in METH:
            T=flagset[m];acc={k:[0,0] for k in ["hour 1","hours 2-3","hours 4-12","hours 13+"]}
            for i in long_i:
                b=ev.building_id[i];s=ev.s[i];e=ev.e[i];vtb=vt.get(b)
                for lab,lo,hi in [("hour 1",0,0),("hours 2-3",1,2),("hours 4-12",3,11),("hours 13+",12,10**6)]:
                    a0=s+lo;a1=min(e,s+hi)
                    if a0>e: continue
                    nvld=np.searchsorted(vt[b],a1,"right")-np.searchsorted(vt[b],a0,"left")
                    t=T.get(b);nfl=0 if t is None else np.searchsorted(t,a1,"right")-np.searchsorted(t,a0,"left")
                    acc[lab][0]+=nfl;acc[lab][1]+=nvld
            for lab,(a,n) in acc.items(): pos.append(dict(method=METH[m],position=lab,flagged_hours=int(a),valid_hours=int(n),flag_rate_pct=round(100*a/n,1) if n else np.nan))
R=pd.DataFrame(rows);D=pd.DataFrame(diffs);P=pd.DataFrame(pos)
R.to_csv(OUT_DIR/"persistence_recall_by_length.csv",index=False);D.to_csv(OUT_DIR/"persistence_differences.csv",index=False);P.to_csv(OUT_DIR/"persistence_flag_rate_by_position.csv",index=False)
pd.set_option("display.width",250)
print(R[R["mode"]=="per_building"].pivot(index="method",columns="length",values="recall_pct")[LG].to_string())
print(D.to_string(index=False));print(P.pivot(index="method",columns="position",values="flag_rate_pct")[["hour 1","hours 2-3","hours 4-12","hours 13+"]].to_string())
fig,ax=plt.subplots(1,2,figsize=(11,4))
for m in METH:
    s=R[(R["mode"]=="per_building")&(R.method==METH[m])].set_index("length").reindex(LG)
    ax[0].errorbar(range(4),s.recall_pct,yerr=[s.recall_pct-s.ci_low,s.ci_high-s.recall_pct],marker="o",capsize=3,label=METH[m])
    q=P[P.method==METH[m]]; ax[1].plot(q.position,q.flag_rate_pct,marker="o",label=METH[m])
ax[0].set_xticks(range(4));ax[0].set_xticklabels(LG);ax[0].set_xlabel("Event length");ax[0].set_ylabel("Events found (%)");ax[0].set_title("Events found by length, 1% of hours flagged")
ax[1].set_xlabel("Position inside events longer than 12 h");ax[1].set_ylabel("Hours flagged (%)");ax[1].set_title("Do flags fade as a long event continues?")
ax[0].legend(fontsize=7);ax[0].grid(alpha=.3);ax[1].grid(alpha=.3)
plt.tight_layout();plt.savefig(OUT_DIR/"persistence_test.png",dpi=200)
